Imports

In [34]:
import pandas as pd
import numpy as np
import time

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFECV
from sklearn.model_selection import StratifiedKFold, train_test_split

Load data

In [35]:
df = pd.read_pickle("../data/cleaned/NSDUH_2023_clean.pkl")

TARGET = "IRAMDEYR"
WEIGHT = "ANALWT2_C"   # drop it for feature selection

y = df[TARGET]
X = df.drop(columns=[TARGET, WEIGHT])

# Ensure numeric
X = X.apply(pd.to_numeric, errors="coerce").fillna(0)

feature_names = X.columns

Train/Test split

In [36]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

Scale

In [37]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Define logistic regression estimator

In [38]:
estimator = LogisticRegression(
    penalty="l2",
    solver="lbfgs",
    max_iter=2000,
    n_jobs=-1
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

Run RFECV (AUC)

In [39]:
print("Starting RFECV (AUC)")
start = time.time()

rfecv_auc = RFECV(
    estimator=estimator,
    step=0.2,
    cv=cv,
    scoring="roc_auc",
    n_jobs=1
)

rfecv_auc.fit(X_train_scaled, y_train)

print(f"RFECV-AUC completed in {(time.time() - start)/60:.2f} minutes")

Starting RFECV (AUC)
RFECV-AUC completed in 4.70 minutes


Extract and Save RFECV-AUC results

In [40]:
# RFECV ranking_: 1 = selected, 2 = next-best, etc.
auc_ranks = pd.Series(rfecv_auc.ranking_, index=feature_names)

# Top 100 lowest-ranked features
top100_auc = auc_ranks.sort_values().head(100).index.tolist()

pd.Series(top100_auc).to_csv("../results/rfecv_auc_top100.csv", index=False)

print("Saved: rfecv_auc_top100.csv")
print("\nTop 100 RFECV-AUC features:")
for f in top100_auc:
    print(f)

Saved: rfecv_auc_top100.csv

Top 100 RFECV-AUC features:
PREGAGE2
CATAG7
CATAGE
IIEDUHIGHST2
IREDUHIGHST2
IRMARIT
IRSEX
COCLALCUSE
GRSKCOCMON
GRSKMRJWK
RKFQPBLT
RSKYFQTES
RSKYFQDGR
DIFGETHER
DIFGETCRK
RSKBNGWK
SYNMRJFLAG
MILTSPPAR
EDUHIGHCAT
SEXAGE
DRVINDETAG
COMHAPTDL
COMHTELE
COFINANC
COCLDRGUSE
COMHTELE2
COHCTELE2
AGE3
COHCRXDL2
COHCSVHLT2
COMHAPTDL2
ILIMFOTHYR
IISYNSTMREC
IISYNMRJREC
WRKOKRAND
WRKOKPREH
WRKTSTDRG
WRKTSTALC
WRKDRGHLP
WRKDRGEDU
MJDABYR
MJSMKYR
CIGYR
MJCMOTHMON
MJSKNMON
WRKSICKMO
WRKNUMJOB2
WRKSELFEM
IIHH65_2
IRHH65_2
IIKI17_2
IRKI17_2
EDFAM18
KRATOMYR
RCVYSUBPRB
CASUPROB2
MJMTHMON
DAMTFXMON
ECSTMOMON
LSDMON
HALLUCMON
IRTRQNMAGE
IRTRQNMYFU
IIIMFREC
SYNSTMREC
SYNSTMEVR
KRATREC
CAMHRCVR
CAMHPROB
CASURCVR
HLTINMNT
CHAMPUS
IISTMNMINIT
IITRQNMINIT
SALVIAMON
KETMINYR
OXYCNNMYR
CNSANYYR
PSYANYYR
PRXYDATA
IIKRATREC
IMFNDLEVER
CNSNMYR
MESCEVER
PEYOTEEVER
ILLALCFLG
ILLORALC
ILTOBVAPALC
ILTOBALCFG
IIECSTMOYFU
PSYCHFLAG
CDCGMO
ILLEMMON
ILLEMFLAG
MJONLYFLAG
IMFEVER
HVYDRKMON
BNGDR

Run RFECV (Recall)

In [41]:
print("Starting RFECV (Recall)")
start = time.time()

rfecv_recall = RFECV(
    estimator=estimator,
    step=0.2,
    cv=cv,
    scoring="recall",
    n_jobs=1
)

rfecv_recall.fit(X_train_scaled, y_train)

print(f"RFECV-Recall completed in {(time.time() - start)/60:.2f} minutes")

Starting RFECV (Recall)
RFECV-Recall completed in 3.98 minutes


Extract and Save RFECV-Recall

In [42]:
recall_ranks = pd.Series(rfecv_recall.ranking_, index=feature_names)

# Top 100 lowest-ranked ranked features
top100_recall = recall_ranks.sort_values().head(100).index.tolist()

pd.Series(top100_recall).to_csv("../results/rfecv_recall_top100.csv", index=False)

print("Saved: rfecv_recall_top100.csv")
print("\nTop 100 RFECV-Recall features:")
for f in top100_recall:
    print(f)

Saved: rfecv_recall_top100.csv

Top 100 RFECV-Recall features:
PREGAGE2
CATAG7
CATAGE
IIEDUHIGHST2
IREDUHIGHST2
IRMARIT
IRSEX
COCLALCUSE
GRSKCOCMON
GRSKMRJWK
RKFQPBLT
RSKYFQTES
RSKYFQDGR
DIFGETHER
DIFGETCRK
RSKBNGWK
SYNMRJFLAG
MILTSPPAR
EDUHIGHCAT
SEXAGE
DRVINDETAG
COMHAPTDL
COMHTELE
COFINANC
COCLDRGUSE
COMHTELE2
COHCTELE2
AGE3
COHCRXDL2
COHCSVHLT2
COMHAPTDL2
ILIMFOTHYR
IISYNSTMREC
IISYNMRJREC
WRKOKRAND
WRKOKPREH
WRKTSTDRG
WRKTSTALC
WRKDRGHLP
WRKDRGEDU
MJDABYR
MJSMKYR
CIGYR
MJCMOTHMON
MJSKNMON
WRKSICKMO
WRKNUMJOB2
WRKSELFEM
IIHH65_2
IRHH65_2
IIKI17_2
IRKI17_2
EDFAM18
KRATOMYR
RCVYSUBPRB
CASUPROB2
MJMTHMON
DAMTFXMON
ECSTMOMON
LSDMON
HALLUCMON
IRTRQNMAGE
IRTRQNMYFU
IIIMFREC
SYNSTMREC
SYNSTMEVR
KRATREC
CAMHRCVR
CAMHPROB
CASURCVR
HLTINMNT
CHAMPUS
IISTMNMINIT
IITRQNMINIT
SALVIAMON
KETMINYR
OXYCNNMYR
CNSANYYR
PSYANYYR
PRXYDATA
IIKRATREC
IMFNDLEVER
CNSNMYR
MESCEVER
PEYOTEEVER
ILLALCFLG
ILLORALC
ILTOBVAPALC
ILTOBALCFG
IIECSTMOYFU
PSYCHFLAG
CDCGMO
ILLEMMON
ILLEMFLAG
MJONLYFLAG
IMFEVER
HVYDRKMON